# DeepEval Synthesizer

Generate synthetic **goldens** (test data) with DeepEval, organised by source.

- **Single-turn** goldens = one atomic input (+ optional expected output).
- **Conversational (multi-turn)** goldens = a scenario for a whole dialogue.
- Four sources each: **documents**, **contexts**, **seed goldens**, **from scratch**.

## 1 · Install dependencies

In [ ]:
%%capture
%pip install -q deepeval pandas httpx nest_asyncio python-dotenv \
    chromadb tiktoken pypdf langchain-core langchain-community langchain-text-splitters \
    sentence-transformers langchain-huggingface

## 2 · Configuration

1. Imports
2. Azure model
3. Embedding model.

In [ ]:
import os

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv

from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ContextConstructionConfig
from deepeval.dataset import Golden, ConversationalGolden
from deepeval.models import AzureOpenAIModel
from deepeval.models.base_model import DeepEvalBaseEmbeddingModel
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
nest_asyncio.apply()
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"

azure_chat = AzureOpenAIModel(
    model="gpt-5",
    deployment_name="gpt-5",
    api_key="FkrroqazToY6a23AAABACOGKkmt",
    base_url="https://ai-azure.com/",
    api_version="202review",
    generation_kwargs={"reasoning_effort": "minimal"},

)

# Local HF embedder (all-MiniLM-L6-v2, CPU), adapted to DeepEval's embedder interface.
class HFEmbeddingModel(DeepEvalBaseEmbeddingModel):
    def load_model(self):
        return HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={"device": "cpu"},
        )

    def embed_text(self, text):
        return self.model.embed_query(text)

    def embed_texts(self, texts):
        return self.model.embed_documents(texts)

    async def a_embed_text(self, text):
        return self.model.embed_query(text)

    async def a_embed_texts(self, texts):
        return self.model.embed_documents(texts)

    def get_model_name(self):
        return "sentence-transformers/all-MiniLM-L6-v2"


context_config = ContextConstructionConfig(
    embedder=HFEmbeddingModel(),
    critic_model=azure_chat,
    chunk_size=500, chunk_overlap=50,
    max_contexts_per_document=3, max_context_length=3,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Upload source PDF documents

Upload one or more PDF files. These uploaded PDFs are the only source material used for both generation paths.

In [ ]:
from pathlib import Path

UPLOAD_DIR = Path("uploaded_pdfs")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)


def upload_pdf_files(upload_dir: Path = UPLOAD_DIR) -> list[Path]:
    """Upload any number of PDFs in Colab, or use local PDF_PATHS outside Colab."""
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        pdf_paths = [Path(p) for p in globals().get("PDF_PATHS", [])]
        pdf_paths = [p for p in pdf_paths if p.exists() and p.suffix.lower() == ".pdf"]
        if not pdf_paths:
            raise RuntimeError(
                "This notebook is not running in Colab. Set PDF_PATHS to a list of local PDF files, then rerun this cell."
            )
        return pdf_paths

    uploaded = files.upload()  # Select one or many PDFs in the file picker.
    pdf_paths: list[Path] = []

    for filename, content in uploaded.items():
        if not filename.lower().endswith(".pdf"):
            print(f"Skipping non-PDF file: {filename}")
            continue

        output_path = upload_dir / Path(filename).name
        output_path.write_bytes(content)
        pdf_paths.append(output_path)

    if not pdf_paths:
        raise ValueError("No PDF files were uploaded. Please upload at least one .pdf file.")

    return pdf_paths


pdf_paths = upload_pdf_files()
print(f"Uploaded/selected {len(pdf_paths)} PDF file(s).")
for pdf_path in pdf_paths:
    print(f"- {pdf_path.name}")

Saving output_qa.pdf to output_qa.pdf
Uploaded/selected 1 PDF file(s).
- output_qa.pdf


Helper — view conversational (multi-turn) goldens as a DataFrame (nothing is saved to disk).

In [ ]:
def multiturn_df(goldens):
    """Build a DataFrame from conversational (multi-turn) goldens for display."""
    return pd.DataFrame([
        {
            "scenario": g.scenario,
            "expected_outcome": g.expected_outcome,
            "user_description": g.user_description,
            "context": g.context,
        }
        for g in goldens
    ])

## 3 · Generation configs

1. Evolutions - rewrite each item for complexity/realism
2. Filtration - drop/rewrite low-quality items
3. Styling & ConversationSyling - styling used by the *from scratch* generators.

In [ ]:
from deepeval.synthesizer.config import (
    StylingConfig,
    ConversationalStylingConfig,
    EvolutionConfig,
    FiltrationConfig,
)
from deepeval.synthesizer.types import Evolution

evolution_config = EvolutionConfig(
    num_evolutions=3,
    evolutions={
        Evolution.REASONING: 0.1,
        Evolution.MULTICONTEXT: 0.1,
        Evolution.CONCRETIZING: 0.1,
        Evolution.CONSTRAINED: 0.1,
        Evolution.COMPARATIVE: 0.1,
        Evolution.HYPOTHETICAL: 0.1,
        Evolution.IN_BREADTH: 0.4,
    },
)

# critic_model MUST be the Azure model, else it falls back to OpenAI and crashes.
filtration_config = FiltrationConfig(
    synthetic_input_quality_threshold=0.5,
    max_quality_retries=3,
    critic_model=azure_chat,
)

styling = StylingConfig(
    task="Answer customer questions about a retail banking mobile app.",
    scenario="Customers asking about payments, cards, and account settings.",
    input_format="A single natural-language user question.",
    expected_output_format="A concise, helpful support answer.",
)
conv_styling = ConversationalStylingConfig(
    scenario_context="A customer contacting a bank's support chatbot for help.",
    conversational_task="Resolve account and transaction related queries.",
    participant_roles="A frustrated customer and a helpful support assistant.",
)

## 4 · Single-turn goldens

### 4.1 Generate Single-turn goldens from documents
Native `generate_goldens_from_docs` over the uploaded PDFs — DeepEval chunks them (LangChain), embeds each chunk with the local HuggingFace model, and clusters them into contexts (`context_config`).

In [ ]:
synth = Synthesizer(
    model=azure_chat,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
synth.generate_goldens_from_docs(
    document_paths=[str(p) for p in pdf_paths],
    include_expected_output=True,
    max_goldens_per_context=2,
    context_construction_config=context_config,
)

# Keep the contexts DeepEval built from the PDFs for reuse in 4.2.
doc_contexts = [g.context for g in synth.synthetic_goldens if g.context]
synth.to_pandas()

Output()

[Confident AI Synthesizer Log] WARNING: Filtering not applied: Not enough chunks in smallest document

[Confident AI Synthesizer Log] SUCCESS: Successfully deleted: /tmp/deepeval_chroma_d0opfmvm

[Confident AI Synthesizer Log] SUCCESS: Context Construction: Utilizing 3 out of 3 chunks.

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,Compare QualNet PDES emergency rerouting acros...,None,"QualNet’s PDES-based simulations enable rapid,...",[Question 1: what is a qualnet simulator?\nAns...,None,2,4072,"[Hypothetical, In-Breadth, Comparative]",None,0.9,uploaded_pdfs/output_qa.pdf
1,"Compare QualNet’s HLA, DIS, STK, and Socket in...",None,- HLA interface:\n - Purpose: Federate QualNe...,[Question 1: what is a qualnet simulator?\nAns...,None,2,4072,"[In-Breadth, Concretizing, Comparative]",None,0.7,uploaded_pdfs/output_qa.pdf
2,Assess XuanYuan 2.0’s robustness on multilingu...,None,"Based on the provided context, XuanYuan 2.0 is...",[10. XuanYuan 2.0 dataset is a key component i...,None,3,5036,"[In-Breadth, Concretizing, In-Breadth]",None,0.8,uploaded_pdfs/output_qa.pdf
3,"Compare LLaMA‑2, PaLM‑2, ChatGPT‑175B paramete...",None,Here’s a concise comparison of parameter‑effic...,[10. XuanYuan 2.0 dataset is a key component i...,None,3,5036,"[Comparative, In-Breadth, Multi-context]",None,0.2,uploaded_pdfs/output_qa.pdf
4,Contrast ChatGPT’s 175B params with QualNet’s ...,None,- Parameters vs. scale: ChatGPT has 175B param...,[and prompting methods to support different mo...,None,3,5036,"[Comparative, Constrained, Reasoning]",None,0.2,uploaded_pdfs/output_qa.pdf
5,Design an ablation on code-switched LLMs: prom...,None,Here’s a concise ablation study design focused...,[and prompting methods to support different mo...,None,3,5036,"[Comparative, In-Breadth, In-Breadth]",None,0.2,uploaded_pdfs/output_qa.pdf


### 4.2 Generate Single-turn goldens from contexts
Reuses the contexts DeepEval built from the PDFs in **4.1** (the in-memory `doc_contexts`). Run 4.1 first.

In [ ]:
synth = Synthesizer(
    model=azure_chat,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
synth.generate_goldens_from_contexts(contexts=doc_contexts, max_goldens_per_context=2)
synth.to_pandas()

Output()

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,Which Chinese MRC corpora compare to XuanYuan ...,None,"Based on the provided context, only XuanYuan 2...",[10. XuanYuan 2.0 dataset is a key component i...,None,3,5036,"[Constrained, In-Breadth, Concretizing]",None,0.9,None
1,Assess zero-resource dialect adaptation in mod...,None,Key points based on the provided context:\n\n-...,[10. XuanYuan 2.0 dataset is a key component i...,None,3,5036,"[In-Breadth, In-Breadth, In-Breadth]",None,0.7,None
2,"How do PDES, digital twin, GUI (Design/Visuali...",None,"They combine to deliver high-fidelity, scalabl...",[Question 1: what is a qualnet simulator?\nAns...,None,2,4072,"[Concretizing, Multi-context, Concretizing]",None,0.9,None
3,Assess PDES-driven HLA/DIS/STK federation scal...,None,QualNet uses Parallel Discrete Event Simulatio...,[Question 1: what is a qualnet simulator?\nAns...,None,2,4072,"[In-Breadth, Constrained, Constrained]",None,0.9,None
4,Assess adaptive time-warp kernels enabling sub...,None,"Adaptive time‑warp kernels, a form of optimist...",[Question 1: what is a qualnet simulator?\nAns...,None,2,4072,"[In-Breadth, In-Breadth, In-Breadth]",None,0.9,None
5,Evaluate QualNet’s PDES-driven digital twin en...,None,QualNet is well-suited for PDES-driven digital...,[Question 1: what is a qualnet simulator?\nAns...,None,2,4072,"[Reasoning, In-Breadth, In-Breadth]",None,0.7,None
6,Verify ChatGPT’s 175B params and compare LLaMA...,None,- ChatGPT parameters: According to the provide...,[and prompting methods to support different mo...,None,3,5036,"[Constrained, In-Breadth, Reasoning]",None,0.8,None
7,"If node count doubles mid-simulation, how woul...",None,"In QualNet’s PDES-based simulations, scaling a...",[and prompting methods to support different mo...,None,3,5036,"[In-Breadth, In-Breadth, Hypothetical]",None,0.6,None
8,Evaluate XuanYuan 2.0’s resilience to Classica...,None,"Based on the provided context, XuanYuan 2.0 is...",[10. XuanYuan 2.0 dataset is a key component i...,None,3,5036,"[Reasoning, In-Breadth, In-Breadth]",None,1.0,None
9,"Quantify GPT-4’s parameter scale vs PaLM-2, th...",None,- Parameter scale: GPT-4’s exact parameter cou...,[10. XuanYuan 2.0 dataset is a key component i...,None,3,5036,"[Concretizing, Multi-context, In-Breadth]",None,0.2,None


### 4.3 Generate Single-turn goldens from seed goldens
Augment existing `Golden`s. Include `context` so an expected output can be generated.

In [ ]:
seed_goldens = [
    Golden(
        input="How do I reset my banking app password?",
        context=["Users reset passwords from Settings > Security > Reset Password."],
        source_file="Created by own",
    ),
    Golden(
        input="What is the daily transfer limit?",
        context=["The default daily transfer limit is 50,000 unless raised in-app."],
        source_file="Created by own",
    ),
]

synth = Synthesizer(
    model=azure_chat,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
synth.generate_goldens_from_goldens(goldens=seed_goldens, max_goldens_per_golden=2)
synth.to_pandas()

Output()

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,What are the exact taps in the banking app to ...,None,- Open the banking app\n- Go to Settings\n- Ta...,[Users reset passwords from Settings > Securit...,None,1,64,"[Concretizing, Concretizing, Reasoning]",None,0.8,Created by own
1,How do I reset my online banking password in t...,None,You can reset it in the app: go to Settings > ...,[Users reset passwords from Settings > Securit...,None,1,64,"[Hypothetical, In-Breadth, In-Breadth]",None,0.7,Created by own
2,If I need to transfer $120k today in an emerge...,None,"Yes. The default daily transfer limit is $50,0...","[The default daily transfer limit is 50,000 un...",None,1,64,"[Multi-context, In-Breadth, Hypothetical]",None,0.6,Created by own
3,"How do I increase the 50,000 daily transfer li...",None,You can request a higher limit directly in the...,"[The default daily transfer limit is 50,000 un...",None,1,64,"[Multi-context, Constrained, In-Breadth]",None,0.8,Created by own


### 4.4 Generate Single-turn goldens from scratch
Ungrounded — `styling` describes what to generate.

In [ ]:
synth = Synthesizer(
    model=azure_chat,
    styling_config=styling,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
synth.generate_goldens_from_scratch(num_goldens=5)
synth.to_pandas()

Output()

,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,Compare in-bank vs fintech debit activation se...,None,None,None,None,None,None,"[Comparative, Hypothetical, Comparative]",None,None,None
1,Compare niche issuer APIs: real-time auth hold...,None,None,None,None,None,None,"[In-Breadth, In-Breadth, Comparative]",None,None,None
2,Design outage double-charge dispute steps opti...,None,None,None,None,None,None,"[Hypothetical, Reasoning, Constrained]",None,None,None
3,Compare same-day intl ATM limit increases via ...,None,None,None,None,None,None,"[In-Breadth, Constrained, Comparative]",None,None,None
4,How can admins verify FIDO2 hardware key attes...,None,None,None,None,None,None,"[In-Breadth, In-Breadth, Concretizing]",None,None,None
5,Compare in-bank vs fintech debit activation se...,None,None,None,None,None,None,"[Comparative, Hypothetical, Comparative]",None,None,None
6,Compare niche issuer APIs: real-time auth hold...,None,None,None,None,None,None,"[In-Breadth, In-Breadth, Comparative]",None,None,None
7,Design outage double-charge dispute steps opti...,None,None,None,None,None,None,"[Hypothetical, Reasoning, Constrained]",None,None,None
8,Compare same-day intl ATM limit increases via ...,None,None,None,None,None,None,"[In-Breadth, Constrained, Comparative]",None,None,None
9,How can admins verify FIDO2 hardware key attes...,None,None,None,None,None,None,"[In-Breadth, In-Breadth, Concretizing]",None,None,None


## 5 · Conversational goldens

### 5.1 Generate Conversational goldens from documents
Native `generate_conversational_goldens_from_docs` over the uploaded PDFs, using the same HuggingFace embedder + `context_config`.

In [ ]:
synth = Synthesizer(
    model=azure_chat,
    conversational_styling_config=conv_styling,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
conv_docs_goldens = synth.generate_conversational_goldens_from_docs(
    document_paths=[str(p) for p in pdf_paths],
    include_expected_outcome=True,
    max_goldens_per_context=2,
    context_construction_config=context_config,
)

# Keep these contexts for reuse in 5.2.
conv_doc_contexts = [g.context for g in conv_docs_goldens if g.context]
multiturn_df(conv_docs_goldens)

Output()

[Confident AI Synthesizer Log] WARNING: Filtering not applied: Not enough chunks in smallest document

[Confident AI Synthesizer Log] SUCCESS: Successfully deleted: /tmp/deepeval_chroma_u5fm2m7c

[Confident AI Synthesizer Log] SUCCESS: Context Construction: Utilizing 3 out of 3 chunks.

,scenario,expected_outcome,user_description,context
0,A frustrated customer contacts a bank’s suppor...,The assistant and customer identify a misroute...,None,[Question 1: what is a qualnet simulator?\nAns...
1,A frustrated customer contacts a bank’s suppor...,The customer and bank support identify settlem...,None,[Question 1: what is a qualnet simulator?\nAns...
2,A frustrated customer contacts a bank’s late-n...,The assistant verifies the customer’s identity...,None,[10. XuanYuan 2.0 dataset is a key component i...
3,"During a bank’s support chat, a frustrated cus...","The assistant confirms the ¥48,200 transfer is...",None,[10. XuanYuan 2.0 dataset is a key component i...
4,"A frustrated sailor, deployed overseas with in...",The chatbot securely verifies the sailor’s ide...,None,[and prompting methods to support different mo...
5,A frustrated customer contacts a bank’s suppor...,"The customer’s identity is verified, unfamilia...",None,[and prompting methods to support different mo...


### 5.2 Generate Conversational goldens from contexts
Reuses `conv_doc_contexts` from **5.1** (in memory). Run 5.1 first.

In [ ]:
synth = Synthesizer(
    model=azure_chat,
    conversational_styling_config=conv_styling,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
conv_ctx_goldens = synth.generate_conversational_goldens_from_contexts(
    contexts=conv_doc_contexts, max_goldens_per_context=2
)
multiturn_df(conv_ctx_goldens)

Output()

,scenario,expected_outcome,user_description,context
0,A frustrated customer contacts a bank’s suppor...,The assistant and customer identify the errone...,None,[10. XuanYuan 2.0 dataset is a key component i...
1,A frustrated customer contacts the bank’s supp...,"The assistant verifies the customer’s account,...",None,[10. XuanYuan 2.0 dataset is a key component i...
2,A frustrated customer contacts a bank’s suppor...,The customer understands that the revoked e-ma...,None,[Question 1: what is a qualnet simulator?\nAns...
3,A frustrated customer contacts a bank’s suppor...,The assistant verifies which international tra...,None,[Question 1: what is a qualnet simulator?\nAns...
4,A frustrated customer contacts a bank’s suppor...,The assistant verifies the customer’s identity...,None,[and prompting methods to support different mo...
5,A frustrated customer contacts the bank’s supp...,The assistant identifies a system clock drift ...,None,[and prompting methods to support different mo...
6,A frustrated customer chats with a helpful ban...,"The customer’s identity is verified, recent tr...",None,[and prompting methods to support different mo...
7,A frustrated customer contacts the bank’s supp...,The assistant verifies the customer’s identity...,None,[and prompting methods to support different mo...
8,A frustrated customer contacts a bank’s suppor...,"The assistant verifies the customer, confirms ...",None,[10. XuanYuan 2.0 dataset is a key component i...
9,A frustrated customer contacts a bank’s suppor...,The assistant verifies the customer’s identity...,None,[10. XuanYuan 2.0 dataset is a key component i...


### 5.3 Generate Conversational goldens from goldens

In [ ]:
seed_conv_goldens = [
    ConversationalGolden(
        scenario="A customer wants to dispute a duplicate charge on their card.",
        expected_outcome="The assistant files a dispute and gives a reference number.",
        user_description="A worried customer who noticed a double charge.",
        context=["Customers can dispute charges via Settings > Transactions > Report Issue. Disputes generate a reference number within 24 hours."],
    ),
    ConversationalGolden(
        scenario="A customer cannot log in after changing their phone number.",
        expected_outcome="The assistant verifies identity and restores access.",
        user_description="A locked-out customer in a hurry.",
        context=["Identity verification for a locked account requires the registered email and a government ID upload."],
    ),
]

synth = Synthesizer(
    model=azure_chat,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
conv_from_goldens = synth.generate_conversational_goldens_from_goldens(
    goldens=seed_conv_goldens, max_goldens_per_golden=2
)
multiturn_df(conv_from_goldens)

Output()

,scenario,expected_outcome,user_description,context
0,"In a customer support chat, a customer and a b...",The customer and agent confirm the in-app path...,None,[Customers can dispute charges via Settings > ...
1,A nonprofit treasurer contacts a payment platf...,The treasurer learns how to batch similar disp...,None,[Customers can dispute charges via Settings > ...
2,A customer support agent chats with a customer...,"The customer verifies their registered email, ...",None,[Identity verification for a locked account re...
3,"During a customer service call, a merchant loc...",The merchant verifies their identity by confir...,None,[Identity verification for a locked account re...


### 5.4 Generate Conversational goldens from scratch

In [ ]:
synth = Synthesizer(
    model=azure_chat,
    conversational_styling_config=conv_styling,
    evolution_config=evolution_config,
    filtration_config=filtration_config,
)
conv_scratch_goldens = synth.generate_conversational_goldens_from_scratch(num_goldens=3)
multiturn_df(conv_scratch_goldens)

Output()

,scenario,expected_outcome,user_description,context
0,Irate customer probes bank chatbot about BIN s...,None,None,None
1,Customer and bank chatbot compare two payroll ...,None,None,None
2,Customer and banking chatbot investigate phant...,None,None,None


## 6 · Simulation

Drive multi-turn conversations against a chatbot using the conversational goldens from **5.3** (`conv_from_goldens`, in memory). The chatbot under test is a plain Azure LLM (no RAG). Two modes: LLM-driven role-play and a scripted simulation graph.

In [ ]:
from deepeval.test_case import Turn
from deepeval.simulator import ConversationSimulator, SimulationNode

SYSTEM = "You are a helpful customer-support assistant for a retail bank. Keep replies short."


async def model_callback(input: str, turns) -> Turn:
    """Chatbot under test: a plain Azure LLM call (no RAG), history-aware."""
    history = "\n".join(f"{t.role}: {t.content}" for t in turns)
    prompt = f"{SYSTEM}\n\n{history}\nuser: {input}\nassistant:"
    response, _cost = await azure_chat.a_generate(prompt)
    return Turn(role="assistant", content=response)


def conversations_df(test_cases):
    """Flatten simulated conversations (one row per turn) into a DataFrame."""
    rows = [
        {"conversation": i, "turn": j, "role": t.role, "content": t.content}
        for i, tc in enumerate(test_cases)
        for j, t in enumerate(tc.turns)
    ]
    return pd.DataFrame(rows)


# Reuse the conversational goldens generated in 5.3 (in memory).
sim_goldens = conv_from_goldens
print(f"Using {len(sim_goldens)} goldens for simulation.")

Using 4 goldens for simulation.


## 6 · Simulation

Two ways to drive the simulated user turn:

- 6.1  **LLM-driven role-play (default)** — no `simulation_graph` passed. The simulator freely generates each user message from the golden's `scenario`, `user_description`, and conversation history. Flexible, but less predictable turn-to-turn.


In [ ]:
llm_sim = ConversationSimulator(
    model_callback=model_callback,
    simulator_model=azure_chat,
)
llm_convos = llm_sim.simulate(sim_goldens, max_user_simulations=6)
conversations_df(llm_convos)

Output()

,conversation,turn,role,content
0,0,0,user,Hi there—I'm seeing a series of tiny charges o...
1,0,1,assistant,Happy to help. To find them:\n\n- Open the app...
2,0,2,user,Got it—I'm in the sub-account’s Activity with ...
3,0,3,assistant,Thanks—that helps.\n\nPlease capture:\n- Trans...
4,0,4,user,Perfect—where do I find the transaction ID on ...
5,0,5,assistant,- Finding the transaction ID: Open a charge > ...
6,0,6,user,"Thanks—once I submit, what’s the exact path to..."
7,0,7,assistant,Here’s how to track it:\n\n- Path: App > Suppo...
8,0,8,user,"Great—after I submit, what should I expect in ..."
9,0,9,assistant,Here’s what to expect:\n\n- Immediate: Confirm...


- 6.2 **Graph-based (`SimulationNode`)** — a `simulation_graph` is passed in. The user's next message is constrained to a predefined state machine (fixed trajectories, retry budgets, terminal states) instead of a free-form LLM prompt. Rule-based, reproducible, no deviation from the scripted path.

In [ ]:
# Scripted user turns (a fixed dispute flow); the simulator_model routes the edges.
start = SimulationNode(
    action=lambda: "Hi, I was charged twice for the same order.",
    name="open_dispute",
)
details = start.add_node(
    SimulationNode(
        action=lambda: "The order number is 12345, charged on July 1st.",
        name="give_details",
    ),
    when="The assistant asks for details or acknowledges the issue.",
)
details.add_node(
    SimulationNode(
        action=lambda: "Great, thanks for your help!",
        terminal=True,
        name="thanks",
    ),
    when="The assistant acknowledges it will review or follow up.",
)

graph_golden = sim_goldens[0]  # golden supplies scenario/expected_outcome; turns come from the script

graph_sim = ConversationSimulator(
    model_callback=model_callback,
    simulator_model=azure_chat,
    simulation_graph=start,
)
graph_convos = graph_sim.simulate([graph_golden], max_user_simulations=6)
conversations_df(graph_convos)

Output()

,conversation,turn,role,content
0,0,0,user,"Hi, I was charged twice for the same order."
1,0,1,assistant,Sorry about that. I can help.\n\n- Was this a ...
2,0,2,user,"The order number is 12345, charged on July 1st."
3,0,3,assistant,"Thanks. I’ll check, but I need a bit more:\n\n..."
4,0,4,user,"Great, thanks for your help!"
5,0,5,assistant,"Happy to help! To start the dispute, please sh..."
